In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

In [ ]:
train = pd.read_csv("imputed_train.csv")
test = pd.read_csv("imputed_test.csv")

In [ ]:
num_cols = ['TOT_SERUM_ALBUM', 'DAYS_STAT1A', 'DAYS_STAT2', 'DAYS_STAT1B', 'CREAT_TRR', 'HEMO_PA_MN_TRR', 
            'TBILI', 'CPRA', 'CPRA_PEAK', 'HLAMIS', 'AGE_DON', 'BUN_DON', 'CREAT_DON', 'SGOT_DON', 'SGPT_DON', 
            'TBILI_DON', 'HGT_CM_DON_CALC', 'WGT_KG_DON_CALC', 
            'AGE', 'ISCHTIME', 'HGT_CM_CALC', 
            'WGT_KG_CALC', 'PO2', 'LV_EJECT', 'PO2_FIO2_DON', 'PCO2_DON', 'PH_DON', 'HEMATOCRIT_DON']


In [ ]:
cat_cols = ['GENDER', 'SUD_DEATH', 'IMPL_DEFIBRIL', 'INFECT_IV_DRUG_TRR', 'INOTROPES_TRR', 'OTH_LIFE_SUP_TRR', 
             'STEROID', 'TRANSFUSIONS', 'VENT_SUPPORT_TRR', 'VENTILATOR_TRR', 'HBV_SURF_TOTAL', 'CMV_STATUS', 
             'EBV_SEROSTATUS', 'GENDER_DON', 'ANTIHYPE_DON', 'BLOOD_INF_DON', 'OTHER_INF_DON', 
             'PT_DIURETICS_DON', 'PT_STEROIDS_DON', 'PT_T4_DON', 'PULM_INF_DON', 'URINE_INF_DON', 'VASODIL_DON', 
             'CLIN_INFECT_DON', 'HIST_OTH_DRUG_DON', 'CMV_DON', 'DDAVP_DON', 'ARGININE_DON', 'INSULIN_DON', 
             'LIFE_SUP_TRR', 'PRIOR_TH_SURG_TRR', 'PROTEIN_URINE', 'CARDARREST_NEURO', 'EBV_IGG_CAD_DON', 
             'CDC_RISK_HIV_DON', 'INOTROP_SUPPORT_DON'] + ["END_STAT", 
                "ETHCAT", "ETHCAT_DON", "ACADEMIC_LEVEL_TRR", "ACADEMIC_PRG_TRR", 
            "FUNC_STAT_TRR", "MED_COND_TRR", "PRI_PAYMENT_TRR", "COGNITIVE_DEV_TRR", "MOTOR_DEV_TRR", "COD_CAD_DON", 
            "ABO_MAT", "DIAG", "PROC_TY_HR", "TRANSFUS_TERM_DON", "VAD_DEVICE_TY_TRR", "ctr_quartile"]


In [ ]:
new_cat_cols = ["END_STAT", 
                "ETHCAT", "ETHCAT_DON", "ACADEMIC_LEVEL_TRR", "ACADEMIC_PRG_TRR", 
            "FUNC_STAT_TRR", "MED_COND_TRR", "PRI_PAYMENT_TRR", "COGNITIVE_DEV_TRR", "MOTOR_DEV_TRR", "COD_CAD_DON", 
            "ABO_MAT", "DIAG", "PROC_TY_HR", "TRANSFUS_TERM_DON", "VAD_DEVICE_TY_TRR", "ctr_quartile"]
def one_hot_train_test(train_df, test_df, cat_cols):
    train_enc = pd.get_dummies(
        train_df,
        columns=cat_cols,
        drop_first=False,
        dummy_na=False
    )

    test_enc = pd.get_dummies(
        test_df,
        columns=cat_cols,
        drop_first=False,
        dummy_na=False
    )

    # align columns (missing columns → 0)
    train_enc, test_enc = train_enc.align(
        test_enc, join="left", axis=1, fill_value=0
    )

    return train_enc, test_enc


train, test = one_hot_train_test(
    train, test, new_cat_cols
)

In [ ]:
comb = pd.concat([train, test])

In [ ]:
comb['GSTATUS'] = ((comb['GSTATUS'] == 1) & (comb['GTIME'] <= 365)).astype(int)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# ---------- Config ----------
AGE_COL    = "AGE"
SEX_COL    = "GENDER"     # confirm: is 1 = male?
HEIGHT_COL = "HGT_CM_CALC"
WEIGHT_COL = "WGT_KG_CALC"
MALE_VALUE = 0            # UNOS convention; change if different

# Units for continuous variables (shown in the Variable column)
UNITS = {
    "AGE": "years",
    "CREAT_TRR": "mg/dL",
    "CREAT_DON": "mg/dL",
    "TBILI": "mg/dL",
    "TBILI_DON": "mg/dL",
    "BUN_DON": "mg/dL",
    "ISCHTIME": "hours",
    "CPRA": "%",
    "CPRA_PEAK": "%",
    "HGT_CM_CALC": "cm",
    "WGT_KG_CALC": "kg",
    "HGT_CM_DON_CALC": "cm",
    "WGT_KG_DON_CALC": "kg",
    "AGE_DON": "years",
    "TOT_SERUM_ALBUM": "g/dL",
    "HEMO_PA_MN_TRR": "mmHg",
    "LV_EJECT": "%",
    "PO2": "mmHg",
    "PO2_FIO2_DON": "",
    "PCO2_DON": "mmHg",
    "PH_DON": "",
    "HEMATOCRIT_DON": "%",
    "SGOT_DON": "U/L",
    "SGPT_DON": "U/L",
    "DAYS_STAT1A": "days",
    "DAYS_STAT1B": "days",
    "DAYS_STAT2": "days",
    "HLAMIS": "",
    "HAZ": "z-score",
    "WAZ": "z-score",
}

# ---------- Helpers ----------
def format_pvalue(p):
    if p is None or np.isnan(p):
        return "NA"
    if p < 0.001:
        return "<0.001"
    if p >= 0.10:
        return f"{p:.2f}"
    if p >= 0.01:
        return f"{p:.2f}"  # e.g. 0.05, 0.08
    if p >= 0.001:
        return f"{p:.3f}"  # e.g. 0.005, 0.003

def is_normal(vec, alpha=0.05, sample_cap=5000):
    """Shapiro normality test on a (possibly subsampled) numeric vector."""
    v = pd.Series(vec).dropna().astype(float)
    if len(v) < 8:
        return False
    v = v.sample(min(sample_cap, len(v)), random_state=42)
    try:
        _, p = stats.shapiro(v)
    except Exception:
        return False
    return p > alpha

def fmt_continuous(all_vals, g1_vals, g2_vals):
    """
    Returns (all_str, g1_str, g2_str, p_str, approach_used).
    Chooses mean±SD + t-test if normal in the pooled data, else median (IQR) + Mann-Whitney.
    """
    all_v = pd.Series(all_vals).dropna()
    g1_v  = pd.Series(g1_vals).dropna()
    g2_v  = pd.Series(g2_vals).dropna()

    normal = is_normal(all_v)

    if normal:
        all_str = f"{all_v.mean():.2f} ± {all_v.std():.2f}"
        g1_str  = f"{g1_v.mean():.2f} ± {g1_v.std():.2f}"
        g2_str  = f"{g2_v.mean():.2f} ± {g2_v.std():.2f}"
        _, p = stats.ttest_ind(g1_v, g2_v, equal_var=False)
        approach = "mean±SD, t-test"
    else:
        all_str = f"{all_v.median():.2f} ({all_v.quantile(0.25):.2f}–{all_v.quantile(0.75):.2f})"
        g1_str  = f"{g1_v.median():.2f} ({g1_v.quantile(0.25):.2f}–{g1_v.quantile(0.75):.2f})"
        g2_str  = f"{g2_v.median():.2f} ({g2_v.quantile(0.25):.2f}–{g2_v.quantile(0.75):.2f})"
        _, p = stats.mannwhitneyu(g1_v, g2_v, alternative="two-sided")
        approach = "median (IQR), Mann-Whitney"

    return all_str, g1_str, g2_str, format_pvalue(p), approach

def fmt_categorical(all_vals, g1_vals, g2_vals):
    """Proportions with chi-squared for binary variables."""
    prop_all = pd.Series(all_vals).mean() * 100
    prop_g1  = pd.Series(g1_vals).mean() * 100
    prop_g2  = pd.Series(g2_vals).mean() * 100

    # Build a simple 2x2 contingency table from the two group series
    g1 = pd.Series(g1_vals).dropna().astype(int)
    g2 = pd.Series(g2_vals).dropna().astype(int)

    # Counts: [group][value]
    table = np.array([
        [(g1 == 0).sum(), (g1 == 1).sum()],
        [(g2 == 0).sum(), (g2 == 1).sum()],
    ])

    try:
        # Use Fisher's exact for small cells, chi2 otherwise
        if (table < 5).any():
            _, p = stats.fisher_exact(table)
        else:
            _, p, _, _ = stats.chi2_contingency(table)
    except Exception:
        p = np.nan

    return (f"{prop_all:.1f}%", f"{prop_g1:.1f}%", f"{prop_g2:.1f}%", format_pvalue(p))

# ---------- Compute HAZ and WAZ and append to comb ----------
# pip install pygrowup-erknet
from pygrowup import Calculator
calc = Calculator(adjust_height_data=False, adjust_weight_scores=False,
                  include_cdc=True, logger_name='pygrowup')

def compute_z(df, age_col, sex_col, h_col, w_col, male_val=1,
              age_in_years=True, min_age_years=2):
    haz, waz = [], []
    for _, row in df.iterrows():
        try:
            age_val = row[age_col]
            # Skip infants/toddlers where integer-year age is too coarse
            if pd.isna(age_val) or age_val < min_age_years:
                haz.append(np.nan); waz.append(np.nan); continue
            if pd.isna(row[h_col]) or pd.isna(row[w_col]):
                haz.append(np.nan); waz.append(np.nan); continue
            age_months = age_val * 12 if age_in_years else age_val
            sex = "M" if row[sex_col] == male_val else "F"
            h_z = calc.lhfa(row[h_col], age_months, sex)
            w_z = calc.wfa(row[w_col], age_months, sex)
            haz.append(float(h_z) if h_z is not None else np.nan)
            waz.append(float(w_z) if w_z is not None else np.nan)
        except Exception:
            haz.append(np.nan); waz.append(np.nan)
    return np.array(haz), np.array(waz)

haz, waz = compute_z(comb, AGE_COL, SEX_COL, HEIGHT_COL, WEIGHT_COL,
                    male_val=MALE_VALUE, age_in_years=True, min_age_years=2)
comb["HAZ"] = haz
comb["WAZ"] = waz

n_valid_haz = (~np.isnan(haz)).sum()
n_valid_waz = (~np.isnan(waz)).sum()
print(f"HAZ computed for {n_valid_haz} / {len(haz)} patients (≥2 years only)")
print(f"WAZ computed for {n_valid_waz} / {len(waz)} patients (≥2 years only)")


# ---------- Build continuous / categorical variable lists ----------
cat_cols = []
num_cols = []
for col in comb.columns:
    if col in ("GSTATUS", "GTIME"):
        continue
    n_unique = comb[col].nunique()
    if n_unique == 2:
        cat_cols.append(col)
    else:
        num_cols.append(col)

# Make sure HAZ/WAZ are in numeric list
for v in ("HAZ", "WAZ"):
    if v in comb.columns and v not in num_cols:
        num_cols.append(v)


# ---------- Generate the revised table ----------
def generate_baseline_table(df, group_col, continuous_vars, categorical_vars):
    rows = []

    # Re-label groups: 0 = No event, 1 = Event
    g1 = df[df[group_col] == 0]   # No event
    g2 = df[df[group_col] == 1]   # Event

    # Header row for outcome definition
    rows.append({
        "Variable": f"OUTCOME DEFINITION: graft failure = removal, death, or chronic support (last follow-up)",
        "All": "",
        "No event": "",
        "Event": "",
        "p-value": ""
    })

    for var in continuous_vars:
        if var not in df.columns:
            continue
        unit = UNITS.get(var, "")
        label = f"{var}" + (f" ({unit})" if unit else "")
        all_s, g1_s, g2_s, p_s, approach = fmt_continuous(df[var], g1[var], g2[var])
        rows.append({
            "Variable": label,
            "All": all_s,
            "No event": g1_s,
            "Event": g2_s,
            "p-value": p_s
        })

    for var in categorical_vars:
        if var not in df.columns:
            continue
        all_s, g1_s, g2_s, p_s = fmt_categorical(df[var], g1[var], g2[var])
        rows.append({
            "Variable": var,
            "All": all_s,
            "No event": g1_s,
            "Event": g2_s,
            "p-value": p_s
        })

    return pd.DataFrame(rows)

table = generate_baseline_table(comb, "GSTATUS", num_cols, cat_cols)
table = table.sort_values("Variable").reset_index(drop=True)
table.to_csv("Table_1_revised_with_haz_and_waz.csv", index=False)

print("\nFirst 30 rows of revised Table 1:")
print(table.head(30).to_string(index=False))
print(f"\nSaved to: Table_1_revised.csv  ({len(table)} rows)")